# J11b — Cartographie des transports en Île-de-France

## Objectifs

- Cartographier le métro, le RER, le Transilien et le tramway.
- Ajouter les lignes et les arrêts de bus sur un secteur local.
- Superposer les restaurants issus de la BPE.
- Conserver les identifiants des lignes et des arrêts.
- Préparer les analyses locales et les futures cartes à l’IRIS.

## Périmètre

Une carte régionale du réseau ferré et une première étude locale
sur Cergy et Pontoise.

## Interprétation

La carte décrit la localisation des transports et des établissements.

Les tracés seuls ne permettent pas de mesurer la fréquence,
la fréquentation ou le temps de marche.

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

from pathlib import Path
from datetime import datetime, timezone
from html import escape
import hashlib
import json
import re

import numpy as np
import pandas as pd
import geopandas as gpd
import plotly
import plotly.graph_objects as go
from IPython.display import display

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 120)
CONFIG_PLOTLY = {'scrollZoom': True, 'displaylogo': False}
print('Plotly :', plotly.__version__)
print('GeoPandas :', gpd.__version__)


In [ ]:
#Chemin des exports existants

RACINE = Path(os.environ.get('GEOMARKETING_RACINE',
              r'C:\Users\almou\OneDrive\GeoMarketing_IDF'))
FICHIER_COMMUNES = RACINE / 'data/processed/j11/profil_communes_idf_j11.gpkg'
FICHIER_RESTAURANTS = RACINE / 'data/processed/j11/restaurants_points.gpkg'
FICHIER_DETAIL_RESTAURANTS = RACINE / 'data/processed/restaurants_idf_detail_j9b.csv'  # Facultatif
DOSSIER_BPE_NOMS = RACINE / 'data/raw/bpe'
FICHIER_BPE_NOMS = DOSSIER_BPE_NOMS / 'restaurants_idf_2025.csv'  # Adapter si nécessaire (CSV ou ZIP)
COLONNE_LIBELLE_RESTAURANT = None  # Nom exact d'un champ personnalisé, si nécessaire
AFFICHER_NOMS_RESTAURANTS = False  # Noms uniquement dans les infobulles au survol
FICHIER_FERRE = RACINE / 'data/processed/j11b/reseau_ferre_idfm.gpkg'
FICHIER_BUS = RACINE / 'data/processed/j11b/lignes_bus_idfm.gpkg'
FICHIER_ARRETS = RACINE / 'data/processed/j11b/points_arret_idfm.gpkg'
DOSSIER_CARTES = RACINE / 'outputs/cartes/j11b'
DOSSIER_INTERIM = RACINE / 'data/interim/j11b/plotly'
DOSSIER_PROCESSED = RACINE / 'data/processed/j11b/plotly'
for dossier in [DOSSIER_CARTES, DOSSIER_INTERIM, DOSSIER_PROCESSED]:
    dossier.mkdir(parents=True, exist_ok=True)



In [ ]:
#Fonctions communes de cartographie

def exiger_colonnes(table, colonnes, nom):
    absentes = set(colonnes) - set(table.columns)
    if absentes:
        raise ValueError(f'{nom} : colonnes absentes {sorted(absentes)}. '
                         f'Disponibles : {table.columns.tolist()}')


def texte(valeur, secours='Non renseigné'):
    if valeur is None or pd.isna(valeur) or not str(valeur).strip():
        return secours
    return str(valeur).strip()


def nombres(serie):
    valeurs = pd.to_numeric(
        serie.astype('string').str.strip()
        .str.replace('\u202f', '', regex=False)
        .str.replace('\u00a0', '', regex=False)
        .str.replace(' ', '', regex=False)
        .str.replace(',', '.', regex=False),
        errors='coerce',
    )
    tableau = valeurs.to_numpy(dtype=float, na_value=np.nan, copy=True)
    tableau[~np.isfinite(tableau)] = np.nan
    return pd.Series(tableau, index=serie.index)


def empreinte(chemin):
    resultat = hashlib.sha256()
    with chemin.open('rb') as flux:
        for bloc in iter(lambda: flux.read(1024 * 1024), b''):
            resultat.update(bloc)
    return resultat.hexdigest()


def enregistrer_csv(table, chemin):
    table.to_csv(chemin, index=False, encoding='utf-8-sig')


def lire_couche(chemin, nom, types, layer=None):
    if not chemin.is_file():
        raise FileNotFoundError(f'{nom} : fichier absent {chemin}')
    options = {'layer': layer} if layer is not None else {}
    couche = gpd.read_file(chemin, **options)
    if couche.empty or couche.crs is None:
        raise ValueError(f'{nom} : couche vide ou CRS absent.')
    valide = (couche.geometry.notna() & ~couche.geometry.is_empty
              & couche.geometry.is_valid & couche.geometry.geom_type.isin(types))
    if not valide.all():
        raise ValueError(f'{nom} : {int((~valide).sum())} géométries à vérifier '
                         'dans le notebook de préparation avant de cartographier.')
    return couche.to_crs(2154)


def preparer_communes(couche):
    exiger_colonnes(couche, ['CODGEO'], 'Communes')
    couche = couche.copy()
    codes = (couche['CODGEO'].astype('string').str.strip()
             .str.replace(r'\.0$', '', regex=True))
    codes = codes.where(codes.str.fullmatch(r'\d{1,5}', na=False)).str.zfill(5)
    if codes.isna().any() or not codes.is_unique:
        raise ValueError('Codes communaux absents, invalides ou répétés.')
    couche['CODGEO'] = codes
    noms = pd.Series(pd.NA, index=couche.index, dtype='string')
    for colonne in ['NOM_COMMUNE_CONTOUR', 'NOM_COMMUNE', 'LIBELLE']:
        if colonne in couche.columns:
            candidat = couche[colonne].astype('string').str.strip().replace('', pd.NA)
            noms = noms.fillna(candidat)
    couche['NOM_CARTE'] = noms.fillna(codes)
    return couche


def contours_geojson(communes, tolerance_metres=20):
    affichage = communes[['CODGEO', 'geometry']].to_crs(2154).copy()
    if tolerance_metres:
        affichage.geometry = affichage.geometry.simplify(
            tolerance_metres, preserve_topology=True)
    affichage = affichage.to_crs(4326)
    return json.loads(affichage.to_json(na='null', drop_id=True))


def ajouter_contours(fig, communes, couleur='#f1f5f9', nom='Limites communales',
                     largeur=0.5, tolerance_metres=20):
    fig.add_trace(go.Choropleth(
        geojson=contours_geojson(communes, tolerance_metres),
        locations=communes['CODGEO'].tolist(),
        featureidkey='properties.CODGEO',
        z=[0] * len(communes), zmin=0, zmax=1,
        colorscale=[[0, couleur], [1, couleur]],
        showscale=False, showlegend=False, name=nom,
        marker_line_color='#64748b', marker_line_width=largeur,
        text=[escape(texte(nom_commune)) for nom_commune in communes['NOM_CARTE']],
        hovertemplate='%{text}<br>Code commune : %{location}<extra></extra>',
    ))
    return fig


def configurer_figure(fig, titre, emprise, source):
    xmin, ymin, xmax, ymax = emprise.to_crs(4326).total_bounds
    marge_x = max((xmax - xmin) * 0.04, 0.001)
    marge_y = max((ymax - ymin) * 0.04, 0.001)
    # Emprise fixe : masquer une ligne dans la légende ne recentre pas la carte.
    fig.update_geos(
        visible=False, bgcolor='white', projection_type='mercator',
        fitbounds=False, center={'lon': (xmin + xmax) / 2, 'lat': (ymin + ymax) / 2},
        lonaxis={'range': [xmin - marge_x, xmax + marge_x], 'showgrid': False},
        lataxis={'range': [ymin - marge_y, ymax + marge_y], 'showgrid': False},
    )
    fig.update_layout(
        title={'text': titre, 'x': 0.02}, height=800,
        paper_bgcolor='white', template='plotly_white',
        margin={'l': 10, 'r': 10, 't': 75, 'b': 80},
        legend={'title': {'text': 'Couches — cliquer pour masquer'},
                'groupclick': 'toggleitem'},
        annotations=[{'text': source, 'xref': 'paper', 'yref': 'paper',
                      'x': 0, 'y': -0.07, 'xanchor': 'left', 'showarrow': False,
                      'font': {'size': 11, 'color': '#475569'}}],
    )
    return fig


def afficher_et_exporter(fig, chemin):
    # Inclut Plotly.js dans le HTML : aucun compte Plotly ni token.
    fig.write_html(str(chemin), include_plotlyjs=True, full_html=True,
                   config=CONFIG_PLOTLY)
    fig.show(config=CONFIG_PLOTLY)
    print('Carte enregistrée :', chemin)


def selectionner_secteur(communes, codes, marge_metres=3000):
    absents = set(codes) - set(communes['CODGEO'])
    if absents:
        raise ValueError(f'Communes absentes : {sorted(absents)}')
    selection = communes[communes['CODGEO'].isin(codes)].copy()
    contour = selection.dissolve().geometry.iloc[0]
    zone = gpd.GeoDataFrame({'ZONE': ['Affichage']},
                           geometry=[contour.buffer(marge_metres)], crs=2154)
    contexte = gpd.clip(communes, zone, keep_geom_type=True)
    return selection, zone, contexte


from contextlib import ExitStack
from pathlib import Path
from zipfile import ZipFile
import csv

import pandas as pd


COLONNES_NOMS_RESTAURANTS = [
    'NOM_RESTAURANT_CARTE', 'LIBELLE_RESTAURANT', 'LIBELLE_RESTAURANT_CARTE',
    'NOM_RESTAURANT', 'NOM_AFFICHAGE', 'ENSEIGNE', 'ENSEIGNE1ETABLISSEMENT',
    'NOM_COMMERCIAL', 'DENOMINATION_USUELLE',
    'DENOMINATIONUSUELLEETABLISSEMENT', 'NOMRS', 'RAISON_SOCIALE', 'CNOMRS',
]


def nettoyer_libelle_restaurant(serie):
    absences = {'', 'nan', 'none', '<na>', 'null', 'nd', '[nd]', 'n/d', 'n/a', '-',
                '_z', '_u', '_n', 'inconnu', 'non renseigné', 'non renseigne',
                'non diffusé', 'non diffuse', 'nom non disponible',
                'libellé absent dans la source'}
    valeurs = serie.astype('string').str.strip().str.replace(r'\s+', ' ', regex=True)
    return valeurs.mask(valeurs.str.casefold().isin(absences))


def choisir_noms_restaurants(table, colonne_libelle=None):
    table = table.rename(columns=lambda c: str(c).strip().upper())
    if table.columns.duplicated().any():
        raise ValueError('Noms de colonnes répétés après normalisation.')
    candidats = list(COLONNES_NOMS_RESTAURANTS)
    if colonne_libelle:
        candidats.insert(0, str(colonne_libelle).strip().upper())
    noms = pd.Series(pd.NA, index=table.index, dtype='string')
    colonnes = pd.Series(pd.NA, index=table.index, dtype='string')
    for colonne in dict.fromkeys(candidats):
        if colonne not in table.columns:
            continue
        candidat = nettoyer_libelle_restaurant(table[colonne])
        provenance = pd.Series(colonne, index=table.index, dtype='string')
        # CNOMRS est un complément : conserver les deux informations lorsqu'elles diffèrent.
        if colonne == 'NOMRS' and 'CNOMRS' in table.columns:
            complement = nettoyer_libelle_restaurant(table['CNOMRS'])
            ajouter = (candidat.notna() & complement.notna()
                       & candidat.str.casefold().ne(complement.str.casefold()).fillna(False))
            candidat.loc[ajouter] = candidat.loc[ajouter] + ' — ' + complement.loc[ajouter]
            provenance.loc[ajouter] = 'NOMRS+CNOMRS'
        utiliser = noms.isna() & candidat.notna()
        noms.loc[utiliser] = candidat.loc[utiliser]
        colonnes.loc[utiliser] = provenance.loc[utiliser]
    return noms, colonnes


def normaliser_siret_pour_noms(serie):
    valeurs = serie.astype('string').str.strip().str.replace(r'\.0$', '', regex=True)
    # Ne pas transformer une valeur invalide en identifiant par suppression ou ajout de chiffres.
    return valeurs.where(valeurs.str.fullmatch(r'\d{14}', na=False)
                         & valeurs.ne('00000000000000'))


def siret_des_restaurants(table):
    table = table.rename(columns=lambda c: str(c).strip().upper())
    resultat = pd.Series(pd.NA, index=table.index, dtype='string')
    if 'SIRET' in table.columns:
        resultat = normaliser_siret_pour_noms(table['SIRET'])
    for colonne in ['CLE_EQUIPEMENT', 'ID_OBSERVATION']:
        if colonne not in table.columns:
            continue
        # Seuls les formats explicitement identifiables sont interprétés comme un SIRET.
        valeurs = table[colonne].astype('string').str.strip()
        extrait = valeurs.str.extract(r'^(?:SIRET[_:])?(\d{14})$', expand=False)
        candidat = normaliser_siret_pour_noms(extrait)
        conflit = resultat.notna() & candidat.notna() & resultat.ne(candidat)
        if conflit.any():
            raise ValueError('Les SIRET des points et de leurs identifiants se contredisent.')
        resultat = resultat.fillna(candidat)
    return resultat


def inspecter_csv_pour_noms(fichier, membre=None):
    fichier = Path(fichier)
    with ExitStack() as pile:
        if membre is None:
            flux = pile.enter_context(fichier.open('rb'))
        else:
            archive = pile.enter_context(ZipFile(fichier))
            flux = pile.enter_context(archive.open(membre))
        ligne = flux.readline()
    try:
        entete = ligne.decode('utf-8-sig')
        encodage = 'utf-8-sig'
    except UnicodeDecodeError:
        entete = ligne.decode('cp1252')
        encodage = 'cp1252'
    essais = [(sep, next(csv.reader([entete], delimiter=sep))) for sep in [';', ',', '\t']]
    separateur, colonnes = max(essais, key=lambda resultat: len(resultat[1]))
    correspondance = {c: str(c).strip().upper() for c in colonnes}
    if len(correspondance) != len(colonnes) or len(set(correspondance.values())) != len(colonnes):
        raise ValueError(f'Entête avec colonnes répétées : {fichier.name}')
    return {'fichier': fichier, 'membre': membre, 'sep': separateur,
            'encoding': encodage, 'colonnes': correspondance}


def morceaux_csv_pour_noms(source, colonnes):
    selection = [c for c, normalise in source['colonnes'].items() if normalise in colonnes]
    with ExitStack() as pile:
        if source['membre'] is None:
            flux = pile.enter_context(source['fichier'].open('rb'))
        else:
            archive = pile.enter_context(ZipFile(source['fichier']))
            flux = pile.enter_context(archive.open(source['membre']))
        lecteur = pile.enter_context(pd.read_csv(
            flux, sep=source['sep'], encoding=source['encoding'], dtype='string',
            usecols=selection, chunksize=20000, low_memory=True))
        for morceau in lecteur:
            yield morceau.rename(columns=source['colonnes'])


def trouver_bpe_pour_noms(fichier_bpe=None, dossier_bpe=None):
    # Inspecter les entêtes, sans charger les millions de lignes du fichier national.
    if fichier_bpe is not None and Path(fichier_bpe).is_file():
        fichiers = [Path(fichier_bpe)]
    elif dossier_bpe is not None and Path(dossier_bpe).is_dir():
        fichiers = sorted(p for p in Path(dossier_bpe).rglob('*')
                          if p.is_file() and p.suffix.lower() in {'.csv', '.zip'})
    else:
        return None
    valides = []
    for fichier in fichiers:
        if fichier.suffix.lower() == '.zip':
            with ZipFile(fichier) as archive:
                membres = [m for m in archive.namelist() if m.lower().endswith('.csv')]
        elif fichier.suffix.lower() == '.csv':
            membres = [None]
        else:
            raise ValueError('La récupération des noms attend une BPE au format CSV ou ZIP.')
        for membre in membres:
            source = inspecter_csv_pour_noms(fichier, membre)
            if {'SIRET', 'NOMRS', 'TYPEQU'}.issubset(source['colonnes'].values()):
                valides.append(source)
    if len(valides) > 1:
        choix = '\n'.join(str(s['fichier']) + (' : ' + s['membre'] if s['membre'] else '')
                          for s in valides)
        raise ValueError('Plusieurs sources BPE avec noms existent. Définir FICHIER_BPE_NOMS '
                         'au bloc 3 avec le fichier BPE 2025 utilisé au J6 :\n' + choix)
    return valides[0] if valides else None


def libelles_par_siret(source, cibles, colonne_libelle=None):
    disponibles = set(source['colonnes'].values())
    champs = set(COLONNES_NOMS_RESTAURANTS) | {'SIRET', 'TYPEQU', 'AN'}
    if colonne_libelle:
        champs.add(str(colonne_libelle).strip().upper())
    if 'SIRET' not in disponibles or not (champs - {'SIRET', 'TYPEQU', 'AN'}) & disponibles:
        return pd.DataFrame(columns=['LIBELLE', 'CHAMP']).rename_axis('SIRET')
    correspondances = {}
    ambigus = set()
    siret_trouves = set()
    for morceau in morceaux_csv_pour_noms(source, champs):
        if 'TYPEQU' in morceau.columns:
            morceau = morceau[morceau['TYPEQU'].str.strip().str.upper().eq('A504')].copy()
        if 'AN' in morceau.columns:
            morceau = morceau[morceau['AN'].str.strip().eq('2025')].copy()
        cles = normaliser_siret_pour_noms(morceau['SIRET'])
        morceau = morceau.loc[cles.isin(cibles)].copy()
        cles = cles.loc[morceau.index]
        siret_trouves.update(cles.dropna())
        noms, origines = choisir_noms_restaurants(morceau, colonne_libelle)
        for siret, libelle, champ in zip(cles, noms, origines):
            if pd.isna(libelle) or siret in ambigus:
                continue
            if siret in correspondances and correspondances[siret]['LIBELLE'] != libelle:
                ambigus.add(siret)
                del correspondances[siret]
            else:
                correspondances[siret] = {'LIBELLE': libelle, 'CHAMP': champ}
    print('Source de libellés :', source['fichier'], source['membre'] or '')
    print('SIRET trouvés :', len(siret_trouves), '/', len(cibles),
          '— avec libellé unique :', len(correspondances), '— ambigus :', len(ambigus))
    return pd.DataFrame.from_dict(correspondances, orient='index',
                                 columns=['LIBELLE', 'CHAMP']).rename_axis('SIRET')


def preparer_noms_restaurants(points, fichier_detail=None, fichier_bpe=None,
                             dossier_bpe=None, colonne_libelle=None):
    if 'ID_OBSERVATION' not in points.columns:
        raise ValueError('La couche des restaurants doit conserver ID_OBSERVATION.')
    resultat = points.copy()
    ids = resultat['ID_OBSERVATION'].astype('string').str.strip().replace('', pd.NA)
    if ids.isna().any() or not ids.is_unique:
        raise ValueError('Identifiants des restaurants absents ou répétés.')
    noms, colonnes = choisir_noms_restaurants(resultat, colonne_libelle)
    origines = ('POINTS:' + colonnes).astype('string')
    sirets = siret_des_restaurants(resultat)
    sources_noms = []

    if noms.isna().any() and fichier_detail is not None and Path(fichier_detail).is_file():
        source = inspecter_csv_pour_noms(fichier_detail)
        sources_noms.append(str(source['fichier']))
        disponibles = set(source['colonnes'].values())
        if 'ID_OBSERVATION' in disponibles:
            champs = set(COLONNES_NOMS_RESTAURANTS) | {'ID_OBSERVATION', 'SIRET', 'CLE_EQUIPEMENT'}
            if colonne_libelle:
                champs.add(str(colonne_libelle).strip().upper())
            morceaux = []
            for morceau in morceaux_csv_pour_noms(source, champs):
                morceau['ID_OBSERVATION'] = morceau['ID_OBSERVATION'].str.strip()
                morceau = morceau[morceau['ID_OBSERVATION'].isin(set(ids))].copy()
                if not morceau.empty:
                    morceaux.append(morceau)
            if morceaux:
                detail = pd.concat(morceaux, ignore_index=True)
                if detail['ID_OBSERVATION'].duplicated().any():
                    raise ValueError('Identifiants répétés dans le fichier détaillé des restaurants.')
                detail = detail.set_index('ID_OBSERVATION', drop=False)
                noms_csv, champs_csv = choisir_noms_restaurants(detail, colonne_libelle)
                cles_csv = ids.map(siret_des_restaurants(detail)).astype('string')
                conflit = sirets.notna() & cles_csv.notna() & sirets.ne(cles_csv)
                if conflit.any():
                    raise ValueError('Les SIRET du fichier détaillé ne correspondent pas aux points.')
                sirets = sirets.fillna(cles_csv)
                completer = noms.isna()
                noms.loc[completer] = ids.map(noms_csv).loc[completer]
                origines.loc[completer] = ('CSV:' + source['fichier'].name + ':'
                                          + ids.map(champs_csv).loc[completer])
        elif 'SIRET' in disponibles:
            correspondances = libelles_par_siret(source, set(sirets.dropna()), colonne_libelle)
            completer = noms.isna()
            noms.loc[completer] = sirets.map(correspondances['LIBELLE']).loc[completer]
            origines.loc[completer] = ('CSV:' + source['fichier'].name + ':'
                                      + sirets.map(correspondances['CHAMP']).loc[completer])

    if noms.isna().any() and (noms.isna() & sirets.notna()).any():
        source = trouver_bpe_pour_noms(fichier_bpe, dossier_bpe)
        if source is not None:
            sources_noms.append(str(source['fichier']))
            cibles = set(sirets.loc[noms.isna()].dropna())
            correspondances = libelles_par_siret(source, cibles, colonne_libelle)
            completer = noms.isna()
            noms.loc[completer] = sirets.map(correspondances['LIBELLE']).loc[completer]
            origines.loc[completer] = ('BPE:' + source['fichier'].name + ':'
                                      + sirets.map(correspondances['CHAMP']).loc[completer])
        else:
            print('Aucune BPE CSV/ZIP contenant SIRET, NOMRS et TYPEQU trouvée dans', dossier_bpe)

    print('Libellés retrouvés :', int(noms.notna().sum()), '/', len(resultat))
    if len(resultat) and not noms.notna().any():
        if not sirets.notna().any():
            raison = ('Les points et le fichier détaillé ne fournissent aucun SIRET utilisable. '
                      'Conserver SIRET et NOMRS dans les exports de restaurants J9b/J11, '
                      'ou fournir leur correspondance avec ID_OBSERVATION.')
        else:
            raison = ('Vérifier le chemin de la BPE 2025 au bloc 3 et ses valeurs NOMRS/CNOMRS. '
                      'Un SIRET peut être absent de la source, sans libellé ou associé à des noms ambigus.')
        raise ValueError('Aucun libellé de restaurant retrouvé. ' + raison
                         + '\nColonnes des points : ' + str(points.columns.tolist()))
    resultat['SIRET_JOINTURE_NOMS'] = sirets
    resultat['LIBELLE_RESTAURANT'] = noms
    resultat['NOM_RESTAURANT_RENSEIGNE'] = noms.notna()
    resultat['NOM_RESTAURANT_CARTE'] = noms.fillna('Libellé absent dans la source')
    resultat['ORIGINE_NOM_CARTE'] = origines.fillna('NON_DISPONIBLE')
    resultat.attrs['sources_libelles_restaurants'] = list(dict.fromkeys(sources_noms))
    return resultat


def ajouter_observations(fig, points, nom='Restaurants — BPE 2025',
                         couleur='#c2410c', visible=True, afficher_noms=False):
    if points.empty:
        print('Aucune observation géolocalisée à afficher :', nom)
        return 0
    exiger_colonnes(points, ['ID_OBSERVATION', 'CODGEO'], nom)
    points = points.to_crs(4326)
    noms, _ = choisir_noms_restaurants(points)
    if not noms.notna().any():
        raise ValueError('Aucun libellé retrouvé dans ce secteur. Vérifier la source des noms avant la carte.')
    libelles = []
    noms_affiches = []
    statuts = [c for c in points.columns if c.startswith('STATUT')]
    for index, ligne in points.iterrows():
        disponible = pd.notna(noms.loc[index])
        libelle = str(noms.loc[index]) if disponible else 'Libellé absent dans la source'
        noms_affiches.append(escape(libelle) if disponible else '')
        lignes = [f'<b>{escape(libelle)}</b>',
                  'Code commune déclaré : ' + escape(texte(ligne['CODGEO'])),
                  'ID observation : ' + escape(texte(ligne['ID_OBSERVATION']))]
        if statuts:
            lignes.extend(f'{escape(c)} : {escape(texte(ligne[c]))}' for c in statuts)
        else:
            lignes.append('Contrôle géographique : consulter le diagnostic J11')
        libelles.append('<br>'.join(lignes))
    fig.add_trace(go.Scattergeo(
        lon=points.geometry.x.tolist(), lat=points.geometry.y.tolist(),
        mode='markers+text' if afficher_noms else 'markers', name=nom, visible=visible,
        ids=points['ID_OBSERVATION'].astype('string').tolist(),
        marker={'size': 6, 'color': couleur, 'opacity': 0.8},
        text=noms_affiches, textposition='top center',
        textfont={'size': 10, 'color': '#7c2d12'},
        customdata=[[libelle] for libelle in libelles],
        hovertemplate='%{customdata[0]}<extra></extra>',
    ))
    return len(points)


In [ ]:
#Charger et controler les couches

communes = preparer_communes(lire_couche(
    FICHIER_COMMUNES, 'Communes', ['Polygon', 'MultiPolygon'], layer='communes'))
ferre = lire_couche(FICHIER_FERRE, 'Réseau ferré', ['LineString', 'MultiLineString'])
bus = lire_couche(FICHIER_BUS, 'Bus', ['LineString', 'MultiLineString'])
arrets = lire_couche(FICHIER_ARRETS, 'Arrêts', ['Point'])
restaurants = lire_couche(FICHIER_RESTAURANTS, 'Restaurants', ['Point'])
for nom, couche in [('Réseau ferré', ferre), ('Bus', bus)]:
    exiger_colonnes(couche, ['SOURCE', 'ID_OBJET_SOURCE', 'ID_LIGNE_SOURCE',
                             'NOM_LIGNE_SOURCE', 'MODE'], nom)
    if couche['ID_OBJET_SOURCE'].isna().any() or not couche['ID_OBJET_SOURCE'].is_unique:
        raise ValueError(f'{nom} : identifiants d’objets absents ou répétés.')
exiger_colonnes(arrets, ['ID_ARRET_SOURCE', 'NOM_ARRET', 'MODE_ARRET'], 'Arrêts')
if arrets['ID_ARRET_SOURCE'].isna().any() or not arrets['ID_ARRET_SOURCE'].is_unique:
    raise ValueError('Identifiants d’arrêts absents ou répétés.')
exiger_colonnes(restaurants, ['ID_OBSERVATION', 'CODGEO'], 'Restaurants')
if restaurants['ID_OBSERVATION'].isna().any() or not restaurants['ID_OBSERVATION'].is_unique:
    raise ValueError('Identifiants d’observations de restaurants absents ou répétés.')
restaurants = preparer_noms_restaurants(
    restaurants, fichier_detail=FICHIER_DETAIL_RESTAURANTS,
    fichier_bpe=FICHIER_BPE_NOMS, dossier_bpe=DOSSIER_BPE_NOMS,
    colonne_libelle=COLONNE_LIBELLE_RESTAURANT)
display(restaurants[['ID_OBSERVATION', 'SIRET_JOINTURE_NOMS', 'LIBELLE_RESTAURANT',
                     'ORIGINE_NOM_CARTE']].head(10))
display(pd.DataFrame([{'COUCHE': n, 'NB_OBJETS': len(c), 'CRS': str(c.crs)}
                      for n, c in [('Communes', communes), ('Ferré', ferre),
                                   ('Bus', bus), ('Arrêts', arrets), ('Restaurants', restaurants)]]))


In [ ]:
#Fonctions de tracé et d'arrets

COULEURS_MODES = {
    'METRO': '#2563eb', 'RER': '#dc2626', 'TRANSILIEN': '#7c3aed',
    'TRAMWAY': '#059669', 'CABLE': '#0891b2', 'BUS': '#d97706', 'AUTRE': '#64748b',
}
LIBELLES_MODES = {
    'METRO': 'Métro', 'RER': 'RER', 'TRANSILIEN': 'Transilien',
    'TRAMWAY': 'Tramway', 'CABLE': 'Téléphérique', 'BUS': 'Bus', 'AUTRE': 'Autres',
}


def segments_geometrie(geometrie):
    if geometrie is None or geometrie.is_empty:
        return
    if geometrie.geom_type == 'LineString':
        yield geometrie
    elif geometrie.geom_type in ['MultiLineString', 'GeometryCollection']:
        for partie in geometrie.geoms:
            yield from segments_geometrie(partie)


def tableau_coordonnees(geometries):
    longitudes, latitudes = [], []
    for geometrie in geometries:
        for segment in segments_geometrie(geometrie):
            coordonnees = list(segment.coords)
            longitudes.extend([float(c[0]) for c in coordonnees] + [None])
            latitudes.extend([float(c[1]) for c in coordonnees] + [None])
    return longitudes, latitudes


def nom_ligne_client(nom, mode):
    nom = texte(nom, 'Ligne sans nom')
    for prefixe, libelle in [('METRO ', 'Métro '), ('TRAIN ', 'Transilien '),
                            ('TRAM ', 'Tramway '), ('CABLE ', 'Téléphérique ')]:
        if nom.upper().startswith(prefixe):
            return libelle + nom[len(prefixe):]
    return ('Bus ' + nom) if mode == 'BUS' else nom


def ajouter_lignes_plotly(fig, couche, visible=True, tolerance_metres=5):
    if couche.empty:
        return 0
    exiger_colonnes(couche, ['SOURCE', 'ID_OBJET_SOURCE', 'ID_LIGNE_SOURCE',
                             'NOM_LIGNE_SOURCE', 'MODE'], 'Tracés IDFM')
    affichage = couche.to_crs(2154).copy()
    if tolerance_metres:
        affichage.geometry = affichage.geometry.simplify(
            tolerance_metres, preserve_topology=True)
    affichage = affichage.to_crs(4326)
    ids = affichage['ID_LIGNE_SOURCE'].astype('string').str.strip().replace('', pd.NA)
    # Une absence d'identifiant ne fusionne pas plusieurs objets inconnus.
    affichage['CLE_CARTE'] = ids.fillna('OBJET_' + affichage['ID_OBJET_SOURCE'].astype('string'))
    nombre = 0
    for (source, mode, identifiant), groupe in affichage.groupby(
            ['SOURCE', 'MODE', 'CLE_CARTE'], sort=True, dropna=False):
        premiere = groupe.iloc[0]
        couleur = texte(premiere.get('COULEUR'), COULEURS_MODES.get(mode, '#64748b'))
        if not re.fullmatch(r'#[0-9a-fA-F]{6}', couleur):
            couleur = COULEURS_MODES.get(mode, '#64748b')
        nom = nom_ligne_client(premiere['NOM_LIGNE_SOURCE'], mode)
        dates = []
        if 'DATE_REFERENCE_SOURCE' in groupe.columns:
            dates = sorted({texte(d, '') for d in groupe['DATE_REFERENCE_SOURCE'] if texte(d, '')})
        information = ('<b>' + escape(nom) + '</b><br>Mode : '
                       + escape(LIBELLES_MODES.get(mode, str(mode)))
                       + '<br>Identifiant commercial : '
                       + escape(texte(premiere['ID_LIGNE_SOURCE']))
                       + '<br>Source : ' + escape(str(source))
                       + '<br>Dates source : ' + escape(', '.join(dates) or 'Non renseignées'))
        lon, lat = tableau_coordonnees(groupe.geometry)
        fig.add_trace(go.Scattergeo(
            lon=lon, lat=lat, mode='lines', connectgaps=False,
            line={'width': 2 if mode == 'BUS' else 3, 'color': couleur},
            name=nom, legendgroup=str(mode),
            legendgrouptitle={'text': LIBELLES_MODES.get(mode, str(mode))},
            visible=visible, text=[information] * len(lon),
            hovertemplate='%{text}<extra></extra>',
        ))
        nombre += 1
    return nombre


def ajouter_arrets_plotly(fig, couche, bus_visibles=False):
    if couche.empty:
        return 0
    exiger_colonnes(couche, ['ID_ARRET_SOURCE', 'NOM_ARRET', 'MODE_ARRET'], 'Arrêts')
    if couche['ID_ARRET_SOURCE'].isna().any() or not couche['ID_ARRET_SOURCE'].is_unique:
        raise ValueError('Identifiants d’arrêts absents ou répétés.')
    points = couche.to_crs(4326)
    libelles = {'BUS': 'Bus', 'METRO': 'Métro', 'FERROVIAIRE': 'Ferroviaire',
                'TRAMWAY': 'Tramway', 'CABLE': 'Téléphérique', 'AUTRE': 'Autres'}
    for mode, groupe in points.groupby('MODE_ARRET', dropna=False):
        couleur = '#7c3aed' if mode == 'FERROVIAIRE' else COULEURS_MODES.get(mode, '#64748b')
        informations = []
        for _, ligne in groupe.iterrows():
            informations.append(
                '<b>' + escape(texte(ligne['NOM_ARRET'])) + '</b><br>ID arrêt : '
                + escape(texte(ligne['ID_ARRET_SOURCE'])) + '<br>ID zone : '
                + escape(texte(ligne.get('ID_ZONE_ARRET_SOURCE')))
                + '<br>Type : ' + escape(libelles.get(mode, str(mode))))
        fig.add_trace(go.Scattergeo(
            lon=groupe.geometry.x.tolist(), lat=groupe.geometry.y.tolist(), mode='markers',
            name='Points d’arrêt — ' + libelles.get(mode, str(mode)),
            marker={'size': 5, 'color': couleur, 'symbol': 'circle', 'opacity': 0.8},
            visible='legendonly' if mode == 'BUS' and not bus_visibles else True,
            text=informations, hovertemplate='%{text}<extra></extra>',
        ))
    return len(points)


In [ ]:
#Secteur local et marges d'affichage

CODES_COMMUNES = ['95127', '95500']
MARGE_AFFICHAGE_METRES = 3000
communes_selection, zone_affichage, communes_contexte = selectionner_secteur(
    communes, CODES_COMMUNES, MARGE_AFFICHAGE_METRES)
ferre_idf = gpd.clip(ferre, communes[['geometry']], keep_geom_type=True)
ferre_local = gpd.clip(ferre, zone_affichage, keep_geom_type=True)
bus_local = gpd.clip(bus, zone_affichage, keep_geom_type=True)
emprise = zone_affichage.geometry.iloc[0]
arrets_locaux = arrets[arrets.geometry.intersects(emprise)].copy()
restaurants_locaux = restaurants[restaurants.geometry.intersects(emprise)].copy()
if ferre_idf.empty:
    raise ValueError('Aucun tracé ferré dans le périmètre communal.')
display(pd.DataFrame([{'COUCHE': n, 'NB_OBJETS_LOCAUX': len(c)}
                      for n, c in [('Ferré', ferre_local), ('Bus', bus_local),
                                   ('Arrêts', arrets_locaux), ('Restaurants', restaurants_locaux)]]))


In [ ]:
#Inventaire local

lignes_locales = pd.concat([pd.DataFrame(ferre_local.drop(columns='geometry')),
                           pd.DataFrame(bus_local.drop(columns='geometry'))], ignore_index=True)
inventaire = (lignes_locales.groupby(
    ['SOURCE', 'MODE', 'ID_LIGNE_SOURCE', 'NOM_LIGNE_SOURCE'], dropna=False)
    .size().reset_index(name='NB_OBJETS_GEOGRAPHIQUES_DANS_EMPRISE'))
enregistrer_csv(inventaire, DOSSIER_INTERIM / 'inventaire_lignes_emprise.csv')
display(inventaire.head(30))


In [ ]:
#Carte régionale du réseau ferré

carte_regionale = go.Figure()
ajouter_contours(carte_regionale, communes)
nb_traces_regionales = ajouter_lignes_plotly(
    carte_regionale, ferre_idf, tolerance_metres=10)
configurer_figure(carte_regionale, 'Île-de-France — réseau ferré', communes,
                 'Tracés : Île-de-France Mobilités. Contours : référence J11. '
                 'Simplification d’affichage des lignes : 10 m.')
FICHIER_CARTE_REGIONALE = DOSSIER_CARTES / '01_reseau_ferre_idf_plotly.html'
afficher_et_exporter(carte_regionale, FICHIER_CARTE_REGIONALE)
print('Traces de lignes commerciales affichées :', nb_traces_regionales)



In [ ]:
#Carte locale avec bus et arrets

carte_locale = go.Figure()
ajouter_contours(carte_locale, communes_contexte)
ajouter_contours(carte_locale, communes_selection, couleur='rgba(0,0,0,0)', largeur=1.5)
ajouter_lignes_plotly(carte_locale, ferre_local)
ajouter_lignes_plotly(carte_locale, bus_local)
nb_arrets_affichables = ajouter_arrets_plotly(carte_locale, arrets_locaux)
configurer_figure(carte_locale, 'Cergy et Pontoise — lignes et points d’arrêt',
                 zone_affichage, 'Transports : Île-de-France Mobilités. '
                 'Tracés bus : © contributeurs OpenStreetMap (ODbL). '
                 'Marge de 3 km pour affichage, sans fond de tuiles.')
FICHIER_CARTE_LOCALE = DOSSIER_CARTES / '02_transports_cergy_pontoise_plotly.html'
afficher_et_exporter(carte_locale, FICHIER_CARTE_LOCALE)



In [ ]:
#Carte locale des transports et restaurants

carte_restaurants = go.Figure()
ajouter_contours(carte_restaurants, communes_contexte)
ajouter_contours(carte_restaurants, communes_selection, couleur='rgba(0,0,0,0)', largeur=1.5)
ajouter_lignes_plotly(carte_restaurants, ferre_local)
ajouter_lignes_plotly(carte_restaurants, bus_local, visible='legendonly')
ajouter_arrets_plotly(carte_restaurants, arrets_locaux)
nb_restaurants_affiches = ajouter_observations(carte_restaurants, restaurants_locaux,
                                              afficher_noms=AFFICHER_NOMS_RESTAURANTS)
enregistrer_csv(pd.DataFrame(restaurants_locaux[['ID_OBSERVATION', 'CODGEO',
    'SIRET_JOINTURE_NOMS', 'LIBELLE_RESTAURANT', 'NOM_RESTAURANT_CARTE',
    'NOM_RESTAURANT_RENSEIGNE', 'ORIGINE_NOM_CARTE']]),
    DOSSIER_INTERIM / 'noms_restaurants_carte.csv')
configurer_figure(carte_restaurants, 'Cergy et Pontoise — transports et restaurants',
                 zone_affichage, 'Transports : IDFM ; tracés bus : © contributeurs OpenStreetMap (ODbL). '
                 'Restaurants : BPE 2025, contrôle J11. Marge d’affichage : 3 km.')
FICHIER_CARTE_RESTAURANTS = DOSSIER_CARTES / '03_transports_restaurants_cergy_pontoise_plotly.html'
afficher_et_exporter(carte_restaurants, FICHIER_CARTE_RESTAURANTS)



In [ ]:
#Exporter les couches locales

def exporter_gpkg(couche, chemin, layer):
    if couche.empty:
        print('Couche locale vide :', layer)
        return
    if chemin.exists():
        chemin.unlink()
    couche.to_file(chemin, layer=layer, driver='GPKG', index=False)

exporter_gpkg(ferre_local, DOSSIER_PROCESSED / 'reseau_ferre_secteur.gpkg', 'reseau_ferre')
exporter_gpkg(bus_local, DOSSIER_PROCESSED / 'lignes_bus_secteur.gpkg', 'lignes_bus')
exporter_gpkg(arrets_locaux, DOSSIER_PROCESSED / 'points_arret_secteur.gpkg', 'points_arret')



In [ ]:
#Controles et métadonnées

cartes = [carte_regionale, carte_locale, carte_restaurants]
fichiers_cartes = [FICHIER_CARTE_REGIONALE, FICHIER_CARTE_LOCALE, FICHIER_CARTE_RESTAURANTS]
for fig, fichier in zip(cartes, fichiers_cartes):
    assert fig.layout.geo.visible is False
    assert all(t.type in ['choropleth', 'scattergeo'] for t in fig.data)
    assert fichier.is_file() and fichier.stat().st_size > 0
    for trace in fig.data:
        if trace.type == 'scattergeo' and trace.mode == 'lines':
            assert trace.connectgaps is False
            assert len(trace.lon) == len(trace.lat) == len(trace.text)
assert nb_arrets_affichables == len(arrets_locaux)
assert nb_restaurants_affiches == len(restaurants_locaux)
enregistrer_csv(pd.DataFrame({'FICHIER': [str(p) for p in fichiers_cartes]}),
                 DOSSIER_INTERIM / 'cartes_creees.csv')
metadata = {
    'etape': 'J11B_CARTOGRAPHIE_PLOTLY',
    'date_execution_utc': datetime.now(timezone.utc).isoformat(),
    'plotly_version': plotly.__version__, 'fond_de_tuiles': False, 'cle_api_requise': False,
    'crs_calculs': 'EPSG:2154', 'crs_affichage': 'EPSG:4326',
    'codes_communes_etudiees': CODES_COMMUNES,
    'marge_affichage_metres': MARGE_AFFICHAGE_METRES,
    'afficher_noms_restaurants': AFFICHER_NOMS_RESTAURANTS,
    'nb_restaurants_locaux_avec_nom': int(restaurants_locaux['NOM_RESTAURANT_RENSEIGNE'].sum()),
    'fichier_detail_noms': str(FICHIER_DETAIL_RESTAURANTS),
    'fichier_bpe_noms_configure': str(FICHIER_BPE_NOMS),
    'sources_libelles_restaurants': restaurants.attrs.get('sources_libelles_restaurants', []),
    'simplification_lignes_region_metres': 10, 'simplification_lignes_locales_metres': 5,
    'sources': [{'fichier': str(p), 'sha256': empreinte(p)}
                for p in [FICHIER_COMMUNES, FICHIER_FERRE, FICHIER_BUS,
                          FICHIER_ARRETS, FICHIER_RESTAURANTS]
                + [Path(p) for p in restaurants.attrs.get('sources_libelles_restaurants', [])]],
    'dates_reference_bus_observees': sorted({texte(d, '') for d in bus.get(
        'DATE_REFERENCE_SOURCE', pd.Series(dtype='string')) if texte(d, '')}),
    'cartes': [str(p) for p in fichiers_cartes],
    'sources_documentation': [
        'https://data.iledefrance-mobilites.fr/explore/dataset/traces-du-reseau-ferre-idf/',
        'https://data.iledefrance-mobilites.fr/explore/dataset/traces-des-lignes-regulieres-de-bus-en-ile-de-france/',
        'https://data.iledefrance-mobilites.fr/explore/dataset/arrets/'],
    'limites': ['Un tracé traversant une zone ne prouve pas sa desserte.',
                'Un point d’arrêt peut représenter un quai, sans être un accès public.',
                'Les tracés seuls ne fournissent ni fréquentation ni fréquence.',
                'La marge d’affichage ne représente pas un temps de marche.'],
}
(DOSSIER_INTERIM / 'metadata_cartes_plotly.json').write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
print('Cartes et exports créés. Vérifier le secteur connu avant validation finale.')
